# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

This paper answers: **which content pages should the content team prioritize for
refresh this cycle, and why?**

Content teams currently default to "oldest page first" a rule that ignores whether
a page is actually declining. Across the working slice used in this study (151,981 scored pages, March 2026), 32.7%
were observed as currently declining not a rare edge case, over a third of the
inventory, which makes prioritization genuinely valuable. So mis-prioritizing this queue has real cost: wasted editor
hours on pages that don't need attention, and missed pages that do.

The task is framed as a two-stage pipeline:

(1) classify whether a page is declining
(impressions dropping >20% over the last 30 days vs. the prior 30 days), then

(2) rank declining pages by refresh priority. Success is measured as **Precision@50** of the top 50 pages the model flags, how many are genuinely worth a content
editor's time matched to a realistic weekly review capacity.

In [1]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
ALL_MONTHS_PATH = f"{REL}/fact_content_daily_performance/month=*/*.parquet"

scale = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM read_parquet('{ALL_MONTHS_PATH}')
""").df().iloc[0]

print(f"Rows:              {scale['n_rows']:,}")
print(f"Distinct clients:  {scale['n_clients']:,}")
print(f"Distinct pages:    {scale['n_content']:,}")
print(f"Date range:        {scale['min_date']} → {scale['max_date']}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows:              78,835,655
Distinct clients:  70
Distinct pages:    427,292
Date range:        2025-01-27 00:00:00 → 2026-06-30 00:00:00


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source:** `FlyRank/internship-warehouse` on Hugging Face — the daily fact table
`fact_content_daily_performance` (one row per page per day: impressions, clicks,
average position), joined to `dim_content` (page creation date) and `dim_clients`.

**Full warehouse scope:** 78,835,655 rows across 70 clients and 427,292 distinct
pages, spanning 2025-01-27 to 2026-06-30.

**Working slice for this study:** a single mid-panel month, **March 2026**, with
a fixed decision day of **March 15**. Features are built only from rows on or
before the decision day (a "trailing" window); the label looks only at rows
*after* the decision day within the same month. The final month in the warehouse
(2026-06) is kept sealed and untouched, reserved as a genuinely future test month
for later work — not used anywhere in this paper.

After restricting to pages with at least some trailing impressions: **151,981
scored rows across 44 clients**, base decline rate 32.7%.

**Excluded, and why:**
- `imp_after` — the column the label is directly computed from. Including it as
  a feature pushed AUC to 1.000 (the model was reading the answer); dropping it
  brought AUC back to a realistic level. Confirmed by a dedicated leakage test.
- `trend_direction`, `trend_pct` — pre-computed columns that are themselves
  proxies for the outcome being predicted; using them would let the label leak
  in through the back door.
- Raw `ga4_*` columns when `ga4_data_available` is `FALSE` — partial coverage
  across clients, so these are collapsed into a single `has_ga4_data` flag
  instead of raw values that would carry systematic missingness.
- Pages created after the decision day — excluded so `content_age_days` always
  reflects information genuinely available at decision time, never a future fact.

**Public-safety:** every identifier in the warehouse (`client_hash_id`,
`content_hash_id`) is a hash, not a real name, URL, or query — this holds
throughout every notebook and this paper.

In [2]:
DECISION_DAY = '2026-03-15'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT fx.* FROM fx
""").df()

scored = features[features['imp_trailing'] > 0]
print(f"Scored rows:      {len(scored):,}")
print(f"Distinct clients: {scored['client_hash_id'].nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scored rows:      151,981
Distinct clients: 44


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining` = 1 if a page's post-decision-day impressions fall below
80% of its pre-decision-day (trailing) impressions, 0 otherwise. This is a
directly observed outcome, not a proxy — so the task is binary classification,
then ranking by predicted probability.

**Features (6, all point-in-time, nothing from the future):** `imp_trailing`,
`clk_trailing`, `pos_trailing`, `ctr_trailing`, `has_ga4_data`, `content_age_days`.

**Baseline:** a Week-4 hand-written rule — flag a page as risky if it sits in a
top-20 search position but has below-median click-through rate (a "good spot,
losing clicks" signal). Simple, explainable, and the bar the model has to beat.

**Model:** Random Forest (300 trees, max depth 8, min 20 samples/leaf,
`random_state=42`), chosen over Logistic Regression because the two-way
interactions between age, position, and CTR that drive decline don't fit a
single linear boundary — confirmed empirically: RF outperformed Logistic
Regression on the same split (0.56 vs 0.50 precision@50).

**Validation design:** grouped 80/20 split by `client_hash_id`
(`GroupShuffleSplit`, `random_state=42`) — no client appears in both train and
test. A naive random row split was tested first and showed precision@50 = 0.78,
inflated by the model recognizing clients it had already seen; the honest,
client-grouped number is 0.56. This gap is why the grouped split is the one
reported everywhere else in this paper.

**Leakage checks performed** (full detail in `w06_validation_audit.ipynb`):
1. No label-derived columns (e.g. `imp_after`) in the feature set.
2. Timeline check — every feature aggregates rows on/before the decision day;
   the label aggregates rows strictly after it.
3. Ablation test — removing the dominant feature (`content_age_days`) dropped
   ROC-AUC by only 0.048 (0.637 → 0.589), a small, gradual decline rather than
   the collapse toward ~0.5 that a hidden-label leak would produce. This
   confirms `content_age_days` carries genuine signal, not leaked information.

In [3]:
import pandas as pd

# Numbers reproduced from the already-audited runs in w05_model.ipynb and
# w06_validation_audit.ipynb — not retrained here, just summarized for the paper.
methodology_summary = pd.DataFrame([
    {"check": "Naive random split (BEFORE)", "precision_at_50": 0.78, "n_test_clients": 43, "client_overlap": 43},
    {"check": "Client-grouped split (AFTER)", "precision_at_50": 0.56, "n_test_clients": 9,  "client_overlap": 0},
])
print("=== Split honesty check ===")
print(methodology_summary.to_string(index=False))

ablation = pd.DataFrame([
    {"feature_set": "WITH content_age_days",    "precision_at_50": 0.560, "roc_auc": 0.637},
    {"feature_set": "WITHOUT content_age_days", "precision_at_50": 0.300, "roc_auc": 0.589},
])
print("\n=== Leakage ablation: is content_age_days real signal? ===")
print(ablation.to_string(index=False))
print("\nVerdict: gradual AUC drop (0.048), not a collapse -> genuine signal, not leakage.")

=== Split honesty check ===
                       check  precision_at_50  n_test_clients  client_overlap
 Naive random split (BEFORE)             0.78              43              43
Client-grouped split (AFTER)             0.56               9               0

=== Leakage ablation: is content_age_days real signal? ===
             feature_set  precision_at_50  roc_auc
   WITH content_age_days             0.56    0.637
WITHOUT content_age_days             0.30    0.589

Verdict: gradual AUC drop (0.048), not a collapse -> genuine signal, not leakage.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

All numbers below are from the same client-grouped test split (9 held-out clients,
zero overlap with training), so baseline and model are compared apples-to-apples.

| Model                  | Precision@50 | Base rate |
|-------------------------|:------------:|:---------:|
| Baseline rule (Week 4)  | 0.20         | 0.373     |
| Logistic Regression     | 0.50         | 0.373     |
| **Random Forest**       | **0.56**     | 0.373     |

The Random Forest's top-50 is correct 56% of the time — nearly 3x the baseline rule
(0.20) and 1.5x random chance at this base rate (0.373). The gain over Logistic
Regression (0.50 → 0.56) suggests the signal has real non-linear structure the
linear model partially misses.

**What drives the ranking:** `content_age_days` dominates both Gini importance
(0.405, more than double the runner-up) and permutation importance (0.076, roughly
3x the next feature) — confirmed as genuine signal, not leakage, in Section 3's
ablation test.

*[Figure: precision@50 comparison chart — baseline vs model, from
`work/figures/w07_precision_comparison.png`]*
*[Figure: archetype distribution of the ranked action queue — from
`work/figures/w07_archetype_distribution.png`]*

In [4]:
results = pd.DataFrame([
    {"model": "Baseline rule (Week 4)", "precision_at_50": 0.20, "base_rate": 0.373},
    {"model": "Logistic Regression",    "precision_at_50": 0.50, "base_rate": 0.373},
    {"model": "Random Forest",          "precision_at_50": 0.56, "base_rate": 0.373},
])
results["lift_over_baseline"] = (results["precision_at_50"] / 0.20).round(2)
print(results.to_string(index=False))

                 model  precision_at_50  base_rate  lift_over_baseline
Baseline rule (Week 4)             0.20      0.373                 1.0
   Logistic Regression             0.50      0.373                 2.5
         Random Forest             0.56      0.373                 2.8


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.